In [0]:
silver_df = (
    spark.read
    .format("delta")
    .load("/Volumes/citibike/default/layers/silver/rides")
)

In [0]:
from pyspark.sql.functions import *

gold_rides = (
    silver_df
    .withColumn("ride_year", year("ride_date"))
    .withColumn("ride_month", month("ride_date"))
    .withColumn("ride_day_of_month", dayofmonth("ride_date"))
    .withColumn(
        "duration_category",
        when(col("trip_duration_min") < 10, "Short Ride")
        .when(col("trip_duration_min") < 30, "Medium Ride")
        .otherwise("Long Ride")
    )
    .withColumn(
        "time_of_day",
        when(col("ride_hour").between(6,11),"Morning")
        .when(col("ride_hour").between(12,17),"Afternoon")
        .when(col("ride_hour").between(18,22),"Evening")
        .otherwise("Night")
    )
    .withColumn("ride_count", lit(1))
)

In [0]:
gold_rides = gold_rides.select(

    "ride_id",

    "ride_date",
    "ride_year",
    "ride_month",
    "ride_day_of_month",
    "ride_day",
    "ride_hour",
    "time_of_day",

    "member_casual",
    "rideable_type",

    "start_station_id",
    "start_station_name",

    "end_station_id",
    "end_station_name",

    "start_lat",
    "start_lng",

    "end_lat",
    "end_lng",

    "trip_duration_min",
    "duration_category",

    "ride_count"
)

In [0]:
gold_rides.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/citibike/default/layers/gold/gold_rides")

In [0]:
%sql
drop table if exists citibike.default.gold_rides

In [0]:
%sql
CREATE TABLE IF NOT EXISTS citibike.default.gold_rides
USING DELTA
AS SELECT * FROM delta.`/Volumes/citibike/default/layers/gold/gold_rides`

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from citibike.default.gold_rides
limit 10;

ride_id,ride_date,ride_year,ride_month,ride_day_of_month,ride_day,ride_hour,time_of_day,member_casual,rideable_type,start_station_id,start_station_name,end_station_id,end_station_name,start_lat,start_lng,end_lat,end_lng,trip_duration_min,duration_category,ride_count
80AE8783E1C1A6D7,2025-02-07,2025,2,7,Friday,8,Morning,member,electric_bike,7023.04,W 59 St & 10 Ave,6474.11,E 40 St & 5 Ave,40.770513,-73.988038,40.752062307,-73.9816324043,7.966666666666667,Short Ride,1
B146900469A936F5,2025-02-05,2025,2,5,Wednesday,8,Morning,member,electric_bike,5262.09,Madison St & Montgomery St,5703.13,Ave A & E 11 St,40.713126,-73.984844,40.72854745023944,-73.98175925016403,5.916666666666667,Short Ride,1
F23AF554BA57D609,2025-02-13,2025,2,13,Thursday,15,Afternoon,member,electric_bike,4060.09,Carroll St & 5 Ave,4446.05,Dean St & Hoyt St,40.6751622,-73.9814832,40.6864442,-73.98759104,10.8,Medium Ride,1
A8301CC0AB1DBFAB,2025-02-13,2025,2,13,Thursday,6,Morning,member,electric_bike,6022.04,E 16 St & 5 Ave,5679.05,Mercer St & Bleecker St,40.73726186,-73.99238967,40.72706363348306,-73.99662137031554,5.283333333333333,Short Ride,1
87EA0E4F5E9F101A,2025-02-05,2025,2,5,Wednesday,22,Evening,member,electric_bike,5653.12,Spring St & Hudson St,5422.09,Lafayette St & Grand St,40.72583996792375,-74.0076532959938,40.72028,-73.99879,3.6,Short Ride,1
11D8F6945DD53A5F,2025-02-09,2025,2,9,Sunday,9,Morning,member,electric_bike,5458.02,N 5 St & Northside Piers,5944.01,Franklin St & Dupont St,40.719876,-73.963865,40.73564,-73.95866,12.983333333333333,Medium Ride,1
C78649CE22EEC8CF,2025-02-09,2025,2,9,Sunday,17,Afternoon,member,classic_bike,5860.02,Green St & McGuinness Blvd,5944.01,Franklin St & Dupont St,40.73396,-73.95204,40.73564,-73.95866,6.683333333333334,Short Ride,1
EC957D1A94D08E12,2025-02-14,2025,2,14,Friday,8,Morning,member,electric_bike,4157.10,Bergen St & Vanderbilt Ave,3919.12,Eastern Pkwy & Franklin Ave (SW Corner),40.6794388,-73.9680438,40.670529,-73.958222,5.133333333333334,Short Ride,1
D315544B54F32FE9,2025-02-06,2025,2,6,Thursday,20,Evening,member,electric_bike,5379.10,N 6 St & Bedford Ave,5944.01,Franklin St & Dupont St,40.71745169,-73.95850939,40.73564,-73.95866,8.016666666666667,Short Ride,1
35B4AB0224FC8DBA,2025-02-12,2025,2,12,Wednesday,18,Evening,member,classic_bike,6170.02,Vernon Blvd & 50 Ave,5944.01,Franklin St & Dupont St,40.74232744,-73.95411749,40.73564,-73.95866,6.416666666666667,Short Ride,1


In [0]:
%sql
Select count(*)
from citibike.default.gold_rides;

count(*)
3116862
